# REALM (2020)
---
[[paper]](https://arxiv.org/pdf/2004.04906)<br>
REALM = Retrieval-Augmented Language Model

REALM — это модель, предложенная исследователями из Google AI, которая представляет собой предобученную языковую модель (Language Model, LM), дополненную механизмом извлечения документов (Retrieval). Она разработана для улучшения фактической точности генерации текста за счет динамического обращения к обширной базе знаний.

**Контекст**

Большие языковые модели (LLM) демонстрируют впечатляющие способности к генерации связного и грамматически правильного текста. Однако они часто страдают от «галлюцинаций» (hallucinations), когда генерируют фактически неточные или выдуманные утверждения. Это происходит потому, что их знания ограничены данными, на которых они были обучены (parametric memory), и они не могут получать доступ к актуальной информации или обновлять свои знания без полного переобучения. Для задач, требующих точных и своевременных фактов, таких как ответы на вопросы с открытым доменом (Open-domain Question Answering), такая зависимость от внутренней, "замороженной" памяти становится серьезным ограничением.

**Идея метода**

Ключевая идея REALM заключается в том, чтобы преодолеть ограничения параметрической памяти LLM путем интеграции *внешнего, явно индексированного хранилища знаний* непосредственно в процесс инференса и обучения модели. Вместо того чтобы полагаться исключительно на знания, усвоенные во время предобучения, REALM учится динамически извлекать релевантные фрагменты текста из большого текстового корпуса (например, Wikipedia) и использовать их как контекст для генерации ответа. Это позволяет модели быть более фактической, прозрачной и способной адаптироваться к изменяющимся знаниям без необходимости полного переобучения.

**Постановка задачи**

REALM решает задачу **генерации текста, основанной на фактах (fact-grounded text generation)**, с особым акцентом на **ответы на вопросы с открытым доменом (Open-domain Question Answering)**. Цель состоит в том, чтобы, получив вопрос, модель могла генерировать точный и информативный ответ, который подтверждается информацией из обширной, неструктурированной базы знаний.

**Существующие альтернативы**

До появления REALM существовали различные подходы к работе с фактической информацией в языковых моделях:

*   **Стандартные предобученные языковые модели (например, BERT (2018), GPT-2 (2019))**: Эти модели обучаются на огромных текстовых корпусах, пытаясь закодировать все знания в свои параметры. Их архитектура не предусматривает доступа к внешним базам данных во время инференса. Как следствие, они склонны к галлюцинациям и не могут обрабатывать вопросы, требующие знаний, отсутствующих в их обученных параметрах или появившихся после обучения.
*   **Двухэтапные системы Open-domain QA (например, DrQA (2017), DPR (2020))**: Эти системы состоят из двух отдельных компонентов:
    *   **Retriever**: Извлекает N наиболее релевантных документов/пассажей из базы знаний на основе запроса. Примеры включают Sparse Retrieval (BM25) или Dense Retrieval (как в DPR).
    *   **Reader**: Принимает извлеченные документы и исходный запрос, а затем читает их, чтобы сгенерировать точный ответ. Обычно это Transformer-based модель (например, BERT).
    *   **Архитектурное отличие**: В этих системах Retriever и Reader обучаются (или настраиваются) **отдельно**. Retriever обычно не является дифференцируемым относительно задачи Reader'а, что ограничивает их совместную оптимизацию. DPR, хотя и использует Dense Retrieval, все равно является двухфазным, где Retriever предобучен отдельно.
*   **Языковые модели, дополненные графами знаний (Knowledge Graph augmented LMs) (например, K-BERT (2020))**: Эти модели пытаются инкорпорировать структурированные знания из графов знаний (Knowledge Graphs, KG) в процесс обучения или инференса.
    *   **Архитектурное отличие**: Зависимость от ручного курирования и создания KG, а также от их статической природы. REALM использует необработанный текстовый корпус, что обеспечивает большую гибкость и актуальность.

REALM отличается тем, что **интегрирует компонент извлечения (Retrieval) непосредственно в архитектуру языковой модели** и делает его **конечно-дифференцируемым (end-to-end differentiable)**. Это позволяет Retriever'у обучаться совместно с генератором текста, оптимизируясь под общую цель – производство точных ответов.

**Архитектура**

Модель REALM состоит из трех основных компонентов:

1.  **Knowledge Retriever (Извлекатель знаний)**:
    *   Представляет собой Transformer-encoder модель (например, BERT-base), которая индексирует все документы из большой базы знаний (например, Wikipedia) в виде плотных эмбеддингов (dense embeddings).
    *   Каждый документ/пассаж $p_i$ преобразуется в эмбеддинг $E_P(p_i)$. Эти эмбеддинги предварительно рассчитываются и хранятся в индексе ближайших соседей (например, FAISS).
    *   При получении запроса $q$, тот же Knowledge Retriever генерирует эмбеддинг запроса $E_Q(q)$.
    *   Релевантность документа $p_i$ запросу $q$ оценивается через скалярное произведение (dot product) их эмбеддингов: $score(q, p_i) = E_Q(q) \cdot E_P(p_i)$.
2.  **Knowledge-Augmented Encoder (Расширенный энкодер знаний)**:
    *   Это еще одна Transformer-encoder модель (например, BERT-large), которая выступает в роли "читателя" (Reader).
    *   Она принимает на вход исходный запрос $q$ и набор из $K$ наиболее релевантных пассажей $\{p_1, \ldots, p_K\}$, извлеченных Knowledge Retriever'ом.
    *   Эти пассажи и запрос конкатенируются и подаются в модель, которая обрабатывает их совместно с помощью механизма Self-Attention.
    *   На выходе Knowledge-Augmented Encoder генерирует конечный ответ.
3.  **Implicit Retrieval (Неявное извлечение)**:
    *   Ключевая особенность REALM: процесс извлечения пассажей становится "неявным" в том смысле, что Retriever не является дискретным, недифференцируемым шагом. Вместо этого, он моделируется как **вероятностное распределение** над всеми пассажами в базе знаний.
    *   Для данного запроса $q$, вероятность извлечения пассажа $p_i$ вычисляется как $P(p_i|q) \propto \exp(E_Q(q) \cdot E_P(p_i))$.
    *   Во время обучения, вместо того чтобы выбирать *только* top-K пассажей, модель может рассматривать вклад *всех* пассажей через это распределение. Однако на практике, для эффективности, используется приближение с выборкой (sampling) или отсечением по top-K пассажам. Это позволяет градиентам течь через процесс извлечения, оптимизируя Retriever вместе с Reader'ом.

**Алгоритм обучения**

Обучение REALM проходит в два этапа:

1.  **Неконтролируемое предобучение (Unsupervised Pre-training) – "Masked Knowledge Model"**:
    *   **Цель**: Научить Retriever эффективно находить информацию, а Reader — использовать её для восстановления пропущенных частей текста.
    *   **Механизм**: Задача аналогична Masked Language Modeling (MLM) в BERT, но с важным отличием.
        *   Из большого текстового корпуса (например, Wikipedia) выбираются документы.
        *   Из этих документов случайным образом маскируются токены (например, 15% токенов).
        *   Модель *не* предсказывает замаскированные токены напрямую из контекста самого документа. Вместо этого, *Knowledge Retriever* сначала пытается найти $K$ наиболее релевантных пассажей из **всей базы знаний**, которые могли бы содержать информацию для предсказания замаскированных токенов.
        *   Затем *Knowledge-Augmented Encoder* получает исходный документ с маскированными токенами, а также извлеченные $K$ пассажей. Его задача — предсказать замаскированные токены, используя как контекст исходный документ, так и *извлеченные знания*.
        *   **Loss Function**: Оптимизируется сумма log-вероятностей правильного предсказания замаскированных токенов. Эта функция потерь является сквозной и дифференцируемой, что позволяет градиентам течь и к параметрам Knowledge Retriever'а, и к Knowledge-Augmented Encoder'а. Таким образом, Retriever учится извлекать *полезные* для предсказания пассажи, а Encoder — эффективно использовать их.

2.  **Контролируемая донастройка (Supervised Fine-tuning)**:
    *   После предобучения модель донастраивается на конкретных downstream-задачах, таких как Open-domain Question Answering (например, Natural Questions, TriviaQA).
    *   На этом этапе модель получает пары (вопрос, ответ) и учится генерировать правильный ответ, используя тот же механизм извлечения знаний.
    *   **Loss Function**: Обычно используется кросс-энтропийная функция потерь для предсказания токенов ответа.

**Алгоритм инференса**

1.  **Индексация базы знаний**: Все документы (пассажи) из базы знаний (например, Wikipedia) один раз пропускаются через обученный **Knowledge Retriever (Passage Encoder)** для получения их dense embeddings. Эти эмбеддинги сохраняются в высокопроизводительном индексе ближайших соседей (например, FAISS).
2.  **Получение запроса**: Модель получает входящий вопрос $q$.
3.  **Извлечение кандидатов**: Вопрос $q$ пропускается через **Knowledge Retriever (Query Encoder)** для получения его эмбеддинга $E_Q(q)$. Затем этот эмбеддинг используется для поиска $K$ наиболее релевантных пассажей $\{p_1, \ldots, p_K\}$ в ранее созданном индексе ближайших соседей.
4.  **Генерация ответа**: Извлеченные текстовые пассажи $\{p_1, \ldots, p_K\}$ конкатенируются с исходным вопросом $q$ и подаются в **Knowledge-Augmented Encoder**. Encoder обрабатывает эту объединенную последовательность и генерирует конечный ответ.

**Результаты**

REALM был оценен на нескольких широко используемых наборах данных для Open-domain Question Answering, включая Natural Questions и TriviaQA.

*   На Natural Questions, REALM значительно превзошел предыдущие подходы, включая модели на базе BERT без retrieval, увеличив F1-score на 9.5 процентных пункта (с 36.6% до 46.1%) по сравнению с базовой моделью BERT-large.
*   На TriviaQA, REALM также показал существенное улучшение, достигнув F1-score 49.3%, что является значительным приростом по сравнению с моделями, не использующими внешний retrieval.
*   Эти результаты продемонстрировали, что **интеграция дифференцируемого retrieval компонента и совместное его обучение с языковой моделью позволяет достичь гораздо более высокой фактической точности** в задачах, требующих обширных и актуальных знаний.

## 📝 Критический анализ

# REALM (2020)
---
[[paper]](https://arxiv.org/pdf/2004.04906)<br>
REALM = Retrieval-Augmented Language Model

REALM — модель от Google AI, представляющая собой предобученную языковую модель с механизмом извлечения документов для повышения фактической точности текста через динамическое обращение к базе знаний.

## Контекст

Большие языковые модели (LLM) часто генерируют фактически неточные утверждения из-за ограниченности знаний, усвоенных во время обучения. Для задач, требующих актуальных фактов, это является ограничением.

## Идея

REALM интегрирует внешнее хранилище знаний в процесс инференса и обучения, извлекая релевантные фрагменты текста из большого корпуса (например, Wikipedia) для генерации более точных ответов.

## Постановка задачи

REALM решает задачу **fact-grounded text generation**, с акцентом на **Open-domain Question Answering**, генерируя точные ответы, подтвержденные информацией из базы знаний.

## Существующие альтернативы

- **Стандартные LLM (BERT, GPT-2)**: Не имеют доступа к внешним базам данных во время инференса.
- **Двухэтапные системы QA (DrQA, DPR)**: Retriever и Reader обучаются отдельно, что ограничивает совместную оптимизацию.
- **Knowledge Graph augmented LMs (K-BERT)**: Зависимость от статических графов знаний.

REALM интегрирует retrieval в архитектуру модели и делает его **end-to-end differentiable**, позволяя совместное обучение Retriever и Reader.

## Архитектура

1. **Knowledge Retriever**: Transformer-encoder, индексирующий документы в виде dense embeddings и оценивающий их релевантность запросу через dot product.
2. **Knowledge-Augmented Encoder**: Transformer-encoder, обрабатывающий запрос и извлеченные пассажи для генерации ответа.
3. **Implicit Retrieval**: Процесс извлечения моделируется как вероятностное распределение, позволяя градиентам течь через процесс извлечения.

## Алгоритм обучения

1. **Unsupervised Pre-training**: Retriever извлекает релевантные пассажи для предсказания замаскированных токенов, оптимизируя совместно с Encoder.
2. **Supervised Fine-tuning**: Донастройка на задачах QA с использованием кросс-энтропийной функции потерь.

## Алгоритм инференса

1. **Индексация**: Документы индексируются через Knowledge Retriever.
2. **Извлечение кандидатов**: Извлекаются релевантные пассажи для запроса.
3. **Генерация ответа**: Извлеченные пассажи и запрос обрабатываются для генерации ответа.

## Результаты

REALM превзошел предыдущие подходы на Natural Questions и TriviaQA, увеличив F1-score на 9.5 п.п. на Natural Questions и достигнув 49.3% на TriviaQA, демонстрируя улучшение фактической точности.

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример иллюстрации основных концепций REALM с использованием Python и PyTorch.
# Этот код демонстрирует, как можно реализовать основные компоненты REALM, такие как Knowledge Retriever и Knowledge-Augmented Encoder.

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel, BertTokenizer
from faiss import IndexFlatL2

# Установка устройства
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Инициализация токенизатора и модели BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)

# Функция для получения эмбеддингов текста с использованием BERT
def get_embeddings(texts):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1)  # Среднее по всем токенам

# Пример базы знаний (в реальности это будет большой корпус, например, Wikipedia)
knowledge_base = [
    "Python is a programming language.",
    "The capital of France is Paris.",
    "The Great Wall of China is visible from space."
]

# Индексация базы знаний
knowledge_embeddings = get_embeddings(knowledge_base).cpu().numpy()
index = IndexFlatL2(knowledge_embeddings.shape[1])
index.add(knowledge_embeddings)

# Пример запроса
query = "What is the capital of France?"

# Получение эмбеддинга запроса
query_embedding = get_embeddings([query]).cpu().numpy()

# Извлечение наиболее релевантных документов
k = 2  # Количество извлекаемых документов
distances, indices = index.search(query_embedding, k)

# Печать извлеченных документов
print("Query:", query)
print("Retrieved documents:")
for idx in indices[0]:
    print("-", knowledge_base[idx])

# Knowledge-Augmented Encoder (Reader)
class KnowledgeAugmentedEncoder(nn.Module):
    def __init__(self, bert_model):
        super(KnowledgeAugmentedEncoder, self).__init__()
        self.bert_model = bert_model

    def forward(self, query, passages):
        # Конкатенация запроса и извлеченных документов
        combined_texts = [query + " " + passage for passage in passages]
        inputs = tokenizer(combined_texts, return_tensors='pt', padding=True, truncation=True).to(device)
        outputs = self.bert_model(**inputs)
        return outputs.last_hidden_state.mean(dim=1)  # Среднее по всем токенам

# Инициализация Knowledge-Augmented Encoder
knowledge_augmented_encoder = KnowledgeAugmentedEncoder(bert_model).to(device)

# Пример использования Knowledge-Augmented Encoder
retrieved_passages = [knowledge_base[idx] for idx in indices[0]]
output_embeddings = knowledge_augmented_encoder(query, retrieved_passages)

# Печать результата
print("Output embeddings shape:", output_embeddings.shape)
```

### Комментарии к коду:

1. **Knowledge Retriever**: 
   - Используется модель BERT для получения эмбеддингов как для базы знаний, так и для запроса.
   - FAISS используется для индексации и поиска ближайших соседей, что позволяет эффективно извлекать релевантные документы.

2. **Knowledge-Augmented Encoder**:
   - Это компонент, который принимает на вход запрос и извлеченные документы, конкатенирует их и обрабатывает с помощью BERT.
   - В реальной реализации, этот компонент будет обучаться совместно с Knowledge Retriever для оптимизации извлечения и использования знаний.

3. **Индексация и Извлечение**:
   - База знаний индексируется с использованием FAISS, что позволяет быстро находить ближайшие документы по эмбеддингам.
   - Извлечение релевантных документов происходит через поиск ближайших соседей для эмбеддинга запроса.

Этот пример иллюстрирует основные архитектурные особенности REALM, такие как интеграция retrieval компонента и его совместное использование с языковой моделью для генерации более точных и обоснованных ответов.